# PharmGKB EDA: Complete 14-Module Analysis

Comprehensive analysis of ALL PharmGKB modules with df.head() displays for each TSV file.

In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import json
import glob
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
plt.style.use('bmh')
pgkb_path = r'D:\ADE DATASET DOWNLOAD\PharmaGKB_extracted'
eda_path = r'D:\ADE DATASET DOWNLOAD\EDA'

## 1. Data Loading Function

In [ ]:
def load_pgkb_module(subdir, target_file=None):
    dir_path = os.path.join(pgkb_path, subdir)
    if not os.path.exists(dir_path):
        print(f"Directory not found: {subdir}")
        return pd.DataFrame()
    
    if subdir == 'guidelineAnnotations.json':
        all_json = []
        for f in glob.glob(os.path.join(dir_path, "*.json"))[:50]:  # Sample
            with open(f, 'r', encoding='utf-8', errors='ignore') as jf:
                all_json.append(json.load(jf))
        return all_json
    
    if target_file:
        f_path = os.path.join(dir_path, target_file)
        if os.path.exists(f_path):
            return pd.read_csv(f_path, sep='\t', comment='#')
    
    match = next((f for f in os.listdir(dir_path) if f.endswith('.tsv') and 'README' not in f), None)
    if match:
        return pd.read_csv(os.path.join(dir_path, match), sep='\t', comment='#')
        
    return pd.DataFrame()

## 2. Chemicals

In [ ]:
chemicals = load_pgkb_module('chemicals', 'chemicals.tsv')
print(f"Total Chemicals: {len(chemicals):,}")
display(chemicals.head(5))

if not chemicals.empty and 'Type' in chemicals.columns:
    plt.figure(figsize=(10, 6))
    type_counts = chemicals['Type'].value_counts()
    sns.barplot(x=type_counts.values, y=type_counts.index, palette='viridis')
    plt.title('Chemical Entity Types')
    plt.tight_layout()
    plt.show()

## 3. Drugs

In [ ]:
drugs = load_pgkb_module('drugs', 'drugs.tsv')
print(f"Total Drugs: {len(drugs):,}")
display(drugs.head(5))

## 4. Genes

In [ ]:
genes = load_pgkb_module('genes', 'genes.tsv')
print(f"Total Genes: {len(genes):,}")
display(genes.head(5))

if not genes.empty and 'Is VIP' in genes.columns:
    vip_counts = genes['Is VIP'].value_counts()
    plt.figure(figsize=(8, 8))
    plt.pie(vip_counts.values, labels=vip_counts.index, autopct='%1.1f%%')
    plt.title('VIP Gene Distribution')
    plt.tight_layout()
    plt.show()

## 5. Variants

In [ ]:
variants = load_pgkb_module('variants', 'variants.tsv')
print(f"Total Variants: {len(variants):,}")
display(variants.head(5))

## 6. Clinical Variants

In [ ]:
clinical_var = load_pgkb_module('clinicalVariants', 'clinicalVariants.tsv')
print(f"Total Clinical Variants: {len(clinical_var):,}")
display(clinical_var.head(5))

## 7. Phenotypes

In [ ]:
phenotypes = load_pgkb_module('phenotypes', 'phenotypes.tsv')
print(f"Total Phenotypes: {len(phenotypes):,}")
display(phenotypes.head(5))

## 8. Relationships

In [ ]:
relationships = load_pgkb_module('relationships', 'relationships.tsv')
print(f"Total Relationships: {len(relationships):,}")
display(relationships.head(5))

if not relationships.empty and 'Entity1_type' in relationships.columns and 'Entity2_type' in relationships.columns:
    rel_matrix = relationships.groupby(['Entity1_type', 'Entity2_type']).size().unstack(fill_value=0)
    plt.figure(figsize=(12, 8))
    sns.heatmap(rel_matrix, annot=True, fmt='d', cmap='YlOrRd')
    plt.title('Relationship Connectivity Matrix')
    plt.tight_layout()
    plt.show()

## 9. Drug Labels

In [ ]:
labels = load_pgkb_module('drugLabels', 'drugLabels.tsv')
print(f"Total Drug Labels: {len(labels):,}")
display(labels.head(5))

if not labels.empty and 'Source' in labels.columns:
    plt.figure(figsize=(10, 6))
    source_counts = labels['Source'].value_counts()
    plt.pie(source_counts.values, labels=source_counts.index, autopct='%1.1f%%')
    plt.title('Drug Label Sources')
    plt.tight_layout()
    plt.show()

## 10. Guidelines (JSON)

In [ ]:
guidelines = load_pgkb_module('guidelineAnnotations.json')
if guidelines:
    print(f"Total Guidelines Loaded: {len(guidelines)}")
    g_df = pd.DataFrame(guidelines)
    display(g_df.head(5))
    
    if 'cpicLevel' in g_df.columns:
        cpic_counts = g_df['cpicLevel'].value_counts().sort_index()
        plt.figure(figsize=(10, 6))
        sns.barplot(x=cpic_counts.index, y=cpic_counts.values, palette='magma')
        plt.title('CPIC Guideline Levels')
        plt.tight_layout()
        plt.show()

## 11. Occurrences

In [ ]:
occurrences = load_pgkb_module('occurrences', 'occurrences.tsv')
print(f"Total Occurrences: {len(occurrences):,}")
display(occurrences.head(5))

if not occurrences.empty and 'Object Name' in occurrences.columns:
    top_mentions = occurrences['Object Name'].value_counts().head(20)
    plt.figure(figsize=(12, 8))
    sns.barplot(x=top_mentions.values, y=top_mentions.index, palette='flare')
    plt.title('Top 20 Most Mentioned Entities in Literature')
    plt.tight_layout()
    plt.show()

## 12. Summary Annotations

In [ ]:
summary_ann = load_pgkb_module('summaryAnnotations', 'summary_annotations.tsv')
print(f"Total Summary Annotations: {len(summary_ann):,}")
display(summary_ann.head(5))

## 13. Variant Annotations

In [ ]:
var_fa_ann = load_pgkb_module('variantAnnotations', 'var_fa_ann.tsv')
print(f"Total FA Variant Annotations: {len(var_fa_ann):,}")
display(var_fa_ann.head(5))

## 14. Pathways Sample

In [ ]:
pathway_dir = os.path.join(pgkb_path, 'pathways-tsv')
if os.path.exists(pathway_dir):
    p_files = [f for f in os.listdir(pathway_dir) if f.endswith('.tsv')]
    print(f"Total Pathway Files: {len(p_files)}")
    
    pk_count = sum(1 for f in p_files if 'Pharmacokinetics' in f)
    pd_count = sum(1 for f in p_files if 'Pharmacodynamics' in f)
    
    plt.figure(figsize=(8, 6))
    plt.bar(['PK Pathways', 'PD Pathways', 'Other'], [pk_count, pd_count, len(p_files)-pk_count-pd_count])
    plt.title('Curated Biological Pathway Focus')
    plt.tight_layout()
    plt.show()
    
    sample_pathway = pd.read_csv(os.path.join(pathway_dir, p_files[0]), sep='\t')
    print(f"\nSample pathway ({p_files[0]}):")
    display(sample_pathway.head(5))

## 15. Summary Report

In [ ]:
summary = {
    'Module': ['Chemicals', 'Drugs', 'Genes', 'Variants', 'Clinical Variants', 'Phenotypes', 
               'Relationships', 'Drug Labels', 'Guidelines', 'Occurrences', 'Summary Ann', 'Variant Ann'],
    'Records': [len(chemicals), len(drugs), len(genes), len(variants), len(clinical_var), 
                len(phenotypes), len(relationships), len(labels), len(guidelines) if guidelines else 0, 
                len(occurrences), len(summary_ann), len(var_fa_ann)]
}
summary_df = pd.DataFrame(summary)
display(summary_df)

plt.figure(figsize=(12, 6))
sns.barplot(data=summary_df, x='Records', y='Module', palette='rocket')
plt.title('PharmGKB Module Coverage')
plt.tight_layout()
plt.show()

print("\nPharmGKB Analysis Complete!")